# Импорт библиотек
Импортируются необходимые библиотеки для работы с chunking, базой данных chromadb, pandas и sentence-transformers.

In [ ]:
%%capture
!pip install chonkie[all] chonkie optuna qdrant_client mlflow

In [ ]:
from chonkie import RecursiveChunker, Visualizer
from chromadb import PersistentClient
from chromadb.api.models import Collection
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# from chonkie.logger import configure_logging
# configure_logging("off")

import numpy as np
import scipy
import mlflow
import uuid
from qdrant_client import QdrantClient
from qdrant_client import models
import os

# Обёртка для коллекции ChromaDB
Определяется класс-обёртка и функция для удобной работы с коллекциями ChromaDB, включая автоматическое удаление коллекции после выхода из контекста.

In [ ]:
class ChromaCollectionWrapper:
    def __init__(self, name, **kwargs):
        self.name = name
        self.kwargs = kwargs
        self.client = PersistentClient("./data/chroma")
        self.collection = self.client.get_or_create_collection(name, **kwargs)

    def __enter__(self) -> Collection.Collection:
        return self.collection

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("for debug")
        self.client.delete_collection(self.name)

    def __getattr__(self, item):
        return getattr(self.collection, item)

def chroma_collection(name, **kwargs):
    '''
    Универсальный способ получить коллекцию chromadb.
    Можно использовать как с with, так и без него.
    Если используется с with, коллекция удаляется после выхода.
    '''
    return ChromaCollectionWrapper(name, **kwargs)

os.environ['MLFLOW_TRACKING_USERNAME'] = 'admin'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'SuperSecurePassword'

# Адрес mlflow
mlflow.set_tracking_uri("http://109.207.175.83:5000/")
mlflow.set_experiment("alpha_team")

# Клиент Qdrant
client = QdrantClient(
    url="http://109.207.175.83:6333",
    api_key="f860f5aef884d8e282f1a8cc9ef71557o",
)


/tmp/ipython-input-3903051171.py:34: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(


# Функция оценки качества поиска
Функция evaluate вычисляет метрику MRR для поиска по embedding'ам.

In [ ]:
def evaluate_old(embedding_model, questions, collection, total_docs, conf_level=0.95, prompt_name='default', replace_abb=False):
  """
  Выполняет оценку полученных документов на основе MRR (mean reciprocal rank) метрики

  Аргументы:
      embedding_model: модель эмбеддингов
      questions: список вопросов для оценки полученных документов
      collection: база данных, содержащая векторизованных предстваления документов или их чанков
      total_docs: общее количество документов, на их осное расчитывается обратный ранг

  Возвращает:
      df: DataFrame, содержащий список вопросов, позицию первого релевантного документа/чанка, обратный ранг, значение метрики MRR и позиции всех чанков
  """
  columns = ["question", "position", "score", "position_list"]
  row_data = []

  for _, row in questions.iterrows():
    # question = row["question"]
    # y_true = row["page_id"]
    # embedding = embedding_model.embed_documents(question)

    if replace_abb:
        question = replace_abbreviations(row['question'], terms)
    else:
        question = row["question"]
    y_true = row["page_id"]
    if prompt_name == 'default':
        embedding = embedding_model.encode(question)
    else:
        embedding = embedding_model.encode(question, prompt_name=prompt_name)

    results = collection.query(embedding, n_results=total_docs)

    for i, position in enumerate(results["metadatas"][0], start=1):

      if position["page_id"] == y_true:
        data_row = [question, i, 1/i,  [page.values() for page in results["metadatas"][0]]]
        row_data.append(data_row)

        break

    else:
      data_row = [question, 0, 0]
      row_data.append(data_row)

  df = pd.DataFrame(row_data, columns=columns)

  scores = []

  for _ in range(10_000):
      scores.append(np.mean(df.score.sample(10)))

  conf_level = conf_level
  row_mean = np.mean(scores)
  row_std = np.std(scores)
  q = scipy.stats.t.ppf((1+conf_level)/2, len(scores)-1)
  lower = row_mean - q * row_std/len(scores)**0.5
  upper = row_mean + q * row_std/len(scores)**0.5

  df = pd.concat([df, pd.DataFrame(data={"question": "MRR", "position": np.round(row_mean, 3), "score": f"{np.round(lower, 3)}, {np.round(upper, 3)}", "position_list": "-------"}, index=[df.shape[0]])], axis=0)

  return df

import numpy as np
import scipy
from typing import Callable, Tuple

def evaluate(embedding_function : Callable[[str], list[float]], questions, client: QdrantClient, collection_name, total_docs, conf_level=0.95) -> Tuple[pd.DataFrame, float, list[float]]:
  """
  Выполняет оценку полученных документов на основе MRR (mean reciprocal rank) метрики

  Аргументы:
      embedding_function: функция, которая будет возращать нам эмбединги
      questions: список вопросов для оценки полученных документов
      collection: база данных, содержащая векторизованных предстваления документов или их чанков
      total_docs: общее количество документов, на их осное расчитывается обратный ранг

  Возвращает:
      df: DataFrame, содержащий список вопросов, позицию первого релевантного документа/чанка, обратный ранг, позиции всех чанков
      row_mean: MRR
      list[float]: доверительный интервал для MRR
  """
  columns = ["question", "position", "score", "position_list"]
  row_data = []

  for _, row in questions.iterrows():
    question = row["question"]
    y_true = row["page_id"]

    embedding = embedding_function(question)
    result_points = client.query_points(collection_name, embedding, limit=total_docs).points

    for i, position in enumerate(result_points, start=1):

      if position.payload["page_id"] == y_true:
        data_row = [question, i, 1/i,  [page.payload['page_id'] for page in result_points]]
        row_data.append(data_row)

        break

    else:
      data_row = [question, 0, 0]
      row_data.append(data_row)

  df = pd.DataFrame(row_data, columns=columns)

  scores = []

  for _ in range(10_000):
      scores.append(np.mean(df.score.sample(10)))

  conf_level = conf_level
  row_mean = np.mean(scores)
  row_std = np.std(scores)
  q = scipy.stats.t.ppf((1+conf_level)/2, len(scores)-1)
  lower = row_mean - q * row_std/len(scores)**0.5
  upper = row_mean + q * row_std/len(scores)**0.5

  return df, row_mean, [lower, upper]


# Функции предобработки текста
Здесь определены функции для обработки заголовков, извлечения и преобразования таблиц, а также удаления изображений из текста.

In [ ]:
import re

def preprocess_headings(text):
    # Заменяем h1., h2., ... h6. на соответствующее количество #
    def repl(match):
        level = int(match.group(1))
        return '\n' + ('#' * level) + ' '
    return re.sub(r'\bh([1-6])\.\s*', repl, text)


def extract_tables_and_text(text):
    """
    Извлекает все таблицы из текста, преобразует их в markdown и возвращает:
    - текст без таблиц
    - список таблиц в markdown-формате
    """
    # Находит все таблицы по шаблону: строки начинаются и заканчиваются на '|'
    table_pattern = r'(?:\n)?((?:\|.*\|\n?)+)'
    tables = re.findall(table_pattern, text)
    markdown_tables = []

    for table in tables:
        lines = [line.strip() for line in table.strip().split('\n') if line.strip()]
        if not lines:
            continue
        header = lines[0]
        columns = [col.strip() for col in header.strip('|').split('|')]
        separator = '|' + '|'.join(['---'] * len(columns)) + '|'
        markdown_table = '\n'.join([header, separator] + lines[1:])
        markdown_tables.append(markdown_table)

    # Удаляем таблицы из текста
    text_without_tables = re.sub(table_pattern, '', text).strip()

    return text_without_tables, markdown_tables

def replace_tables_with_markdown(text):
    """
    Находит все таблицы в тексте и заменяет их на markdown-таблицы с разделителем после заголовка.
    Возвращает изменённый текст.
    """
    def table_to_markdown(table):
        lines = [line.strip() for line in table.strip().split('\n') if line.strip()]
        if not lines:
            return ''
        header = lines[0]
        columns = [col.strip() for col in header.strip('|').split('|')]
        separator = '|' + '|'.join(['---'] * len(columns)) + '|'
        return '\n'.join([header, separator] + lines[1:])

    table_pattern = r'((?:\|.*\|\n?)+)'
    def replacer(match):
        return table_to_markdown(match.group(1))

    return re.sub(table_pattern, replacer, text)

def remove_image_tags(text):
    """
    Удаляет все конструкции вида {{...расширение}} (png, jpg, jpeg, gif, webp, bmp, svg)
    """
    return re.sub(r'\{\{[^{}]*?\.(png|jpg|jpeg|gif|webp|bmp|svg)\}\}|\{\{undefined\}\}', '', text, flags=re.IGNORECASE)


Добавил функцию для замены ссылок на изображения на [IMG_{здесь мог быть ваш индекс}]. Также функцию восстановления, дальше по коду пока не используется, потому что незачем.

Также функция убирает \n перед ссылками, чтобы чанкеры справлялись лучше

In [ ]:
import re
from typing import Tuple, Dict

def preprocess_images(text: str) -> Tuple[str, Dict[str, str]]:
    """
    Заменяет ссылки на изображения на плейсхолдеры [IMG_N],
    при этом удаляя все \n перед вставкой (до ближайшего не-\n символа).
    """
    pattern = r'!\{[^}]*\}[^!\s]+!?|![^!\s]+!|\{\{[^}]+\}\}'
    replacements = {}
    clean_text = text
    offset = 0  # смещение из-за изменения длины текста

    for match in re.finditer(pattern, text):
        start, end = match.start() - offset, match.end() - offset
        placeholder = f"[IMG_{len(replacements) + 1}]"
        replacements[placeholder] = match.group(0)

        # --- удаляем \n и пробелы перед вставкой ---
        back = start
        while back > 0 and clean_text[back - 1] in ['\n', '\r', ' ']:
            back -= 1
        # но если всё до начала — не трогаем
        if back < start:
            clean_text = clean_text[:back] + placeholder + clean_text[end:]
            offset += (end - back) - len(placeholder)
        else:
            clean_text = clean_text[:start] + placeholder + clean_text[end:]
            offset += (end - start) - len(placeholder)

    return clean_text, replacements


def restore_images(text: str, replacements: Dict[str, str]) -> str:
    """Восстанавливает оригинальные вставки по плейсхолдерам."""
    for placeholder, original in replacements.items():
        text = text.replace(placeholder, original)
    return text


# Загрузка данных и подготовка embedding-модели
Загружаются вопросы и тексты, а также инициализируется модель для получения эмбеддингов.

Ещё немного потыкал другие модельки, пока сложно что-то конкртеное сказать, надо проверять

In [ ]:
!gdown 16a1OP6crVCX1VI6Cukk9eLYYZgd8zAFg -O dataset.zip
!unzip -qq dataset.zip

Downloading...
From (original): https://drive.google.com/uc?id=16a1OP6crVCX1VI6Cukk9eLYYZgd8zAFg
From (redirected): https://drive.google.com/uc?id=16a1OP6crVCX1VI6Cukk9eLYYZgd8zAFg&confirm=t&uuid=091cec13-97a4-4aa8-b697-f7564b19b6f3
To: /content/dataset.zip
100% 160M/160M [00:03<00:00, 44.2MB/s]
replace dataset/questions.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [ ]:
q = pd.read_csv("./dataset/questions.csv")
docs = pd.read_csv("./dataset/texts.csv")
raw_texts = []
for index, row in docs.iterrows():
    with open(f"./dataset/texts/{row['page_id']}.txt", "r") as f:
        raw_texts.append(f.read())

docs["text"] = raw_texts

In [ ]:
terms = pd.read_csv("dataset/terms.csv")

def replace_abbreviations(text: str, terms: pd.DataFrame) -> str:
    """
    Заменяет аббревиатуры в тексте на их полные формулировки, используя DataFrame terms.

    Args:
        text (str): Исходный текст.
        terms (pd.DataFrame): DataFrame с колонками "abbreviation" и "full_form".

    Returns:
        str: Текст с заменёнными аббревиатурами.
    """
    for _, row in terms.iterrows():
        abbreviation = row["name"]
        full_form = row["meaning"]
        # Используем регулярное выражение для точного совпадения аббревиатуры
        text = re.sub(rf'\b{re.escape(abbreviation)}\b', full_form, text)
    return text

# Chunking markdown-текста с RecursiveChunker
Разделение markdown-текста на чанки с помощью RecursiveChunker.

Был 0.96 на 28 документах, сейчас 0.79

In [ ]:
from chonkie import RecursiveChunker

model = "ai-forever/ru-en-RoSBERTa"
# model = "deepvk/USER2-base"
# model = "sergeyzh/rubert-mini-frida"
embeddings = SentenceTransformer(model)

chunker = RecursiveChunker(
    chunk_size = 128
).from_recipe("markdown", lang="en")

collection_name=str(uuid.uuid1())

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embeddings.get_sentence_embedding_dimension(),  # Vector size is defined by used model
        distance=models.Distance.COSINE,
    ),
)


with mlflow.start_run(run_name="chonkie + RecursiveChunker"):
    mlflow.log_param("model", model)
    mlflow.log_param("model_params", embeddings.__dict__)
    mlflow.log_param("collection_name", collection_name)
    mlflow.log_param("chunker_name", str(chunker.__class__))
    mlflow.log_param("chunker_params", chunker.__dict__)

    chunk_lengths = []
    points = []

    for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
        text = row['text']

        for n_chunk, chunk in enumerate(chunker.chunk(text), start=1):
            points.append(models.PointStruct(
                id=len(points) + 1
                , vector=embeddings.encode(chunk.text)
                , payload={"page_id": row["page_id"], "chunk_id": n_chunk, "chunk": chunk.text}
                ))

            chunk_lengths.append(len(chunk.text))


    mlflow.log_metric("total_chunks", len(chunk_lengths))
    mlflow.log_metric("min_chunk_length", min(chunk_lengths))
    mlflow.log_metric("max_chunk_length", max(chunk_lengths))
    mlflow.log_metric("avg_chunk_length", sum(chunk_lengths)/len(chunk_lengths))

    client.upload_points(
        collection_name=collection_name,
        points=points
    )

    def embeddings_function(question: str) -> list[float]:
        return embeddings.encode(question).tolist()

    # --- Оценка разбиения ---
    eval_df, mrr, mrr_ci = evaluate(embeddings_function, q, client, collection_name, len(docs))
    # Сохраняем таблицу как артефакт
    # mlflow.log_table(eval_df, "eval_df.json")
    # Логируем метрику MRR и доверительный интервал
    mlflow.log_metric("MRR", mrr)
    mlflow.log_metric("MRR_lower", float(mrr_ci[0]))
    mlflow.log_metric("MRR_upper", float(mrr_ci[1]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

v1.schema.json: 0.00B [00:00, ?B/s]

markdown_en.json: 0.00B [00:00, ?B/s]

100%|██████████| 172/172 [01:01<00:00,  2.81it/s]


🏃 View run chonkie + RecursiveChunker at: http://109.207.175.83:5000/#/experiments/2/runs/afa07718deed4d30b45df00fb283cadd
🧪 View experiment at: http://109.207.175.83:5000/#/experiments/2


In [ ]:
from chonkie import RecursiveChunker

# model = "ai-forever/ru-en-RoSBERTa"
model = "deepvk/USER2-base"
# model = "sergeyzh/rubert-mini-frida"
embeddings = SentenceTransformer(model)

chunker = RecursiveChunker(
    chunk_size = 128
).from_recipe("markdown", lang="en")

collection_name=str(uuid.uuid1())

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embeddings.get_sentence_embedding_dimension(),  # Vector size is defined by used model
        distance=models.Distance.COSINE,
    ),
)


with mlflow.start_run(run_name="chonkie + RecursiveChunker + vk_embeds"):
    mlflow.log_param("model", model)
    mlflow.log_param("model_params", embeddings.__dict__)
    mlflow.log_param("collection_name", collection_name)
    mlflow.log_param("chunker_name", str(chunker.__class__))
    mlflow.log_param("chunker_params", chunker.__dict__)

    chunk_lengths = []
    points = []

    for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
        text = row['text']

        for n_chunk, chunk in enumerate(chunker.chunk(text), start=1):
            points.append(models.PointStruct(
                id=len(points) + 1
                , vector=embeddings.encode(chunk.text)
                , payload={"page_id": row["page_id"], "chunk_id": n_chunk, "chunk": chunk.text}
                ))

            chunk_lengths.append(len(chunk.text))


    mlflow.log_metric("total_chunks", len(chunk_lengths))
    mlflow.log_metric("min_chunk_length", min(chunk_lengths))
    mlflow.log_metric("max_chunk_length", max(chunk_lengths))
    mlflow.log_metric("avg_chunk_length", sum(chunk_lengths)/len(chunk_lengths))

    client.upload_points(
        collection_name=collection_name,
        points=points
    )

    def embeddings_function(question: str) -> list[float]:
        return embeddings.encode(question).tolist()

    # --- Оценка разбиения ---
    eval_df, mrr, mrr_ci = evaluate(embeddings_function, q, client, collection_name, len(docs))
    # Сохраняем таблицу как артефакт
    # mlflow.log_table(eval_df, "eval_df.json")
    # Логируем метрику MRR и доверительный интервал
    mlflow.log_metric("MRR", mrr)
    mlflow.log_metric("MRR_lower", float(mrr_ci[0]))
    mlflow.log_metric("MRR_upper", float(mrr_ci[1]))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

  0%|          | 0/172 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/backends/cuda/__init__.py:131: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return torch._C._get_cublas_allow_tf32()
W1124 18:21:56.166000 1380 torch/_inductor/utils.py:1558] [1/0_1] Not enough SMs to use max_autotune_gemm mode
100%|██████████| 172/172 [01:01<00:00,  2.77it/s]


🏃 View run chonkie + RecursiveChunker + vk_embeds at: http://109.207.175.83:5000/#/experiments/2/runs/29b65d05c6cb487eb3d023b35662674d
🧪 View experiment at: http://109.207.175.83:5000/#/experiments/2


# Chunking markdown-текста с SemanticChunker
Разделение markdown-текста на чанки с помощью SemanticChunker.

In [ ]:
from chonkie import SemanticChunker, AutoEmbeddings

model = "ai-forever/ru-en-RoSBERTa"
# model = "deepvk/USER2-base"
# model = "sergeyzh/rubert-mini-frida"
embeddings = SentenceTransformer(model)

chunker = SemanticChunker(
    embedding_model = model,
    chunk_size = 256
)

collection_name=str(uuid.uuid1())

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embeddings.get_sentence_embedding_dimension(),  # Vector size is defined by used model
        distance=models.Distance.COSINE,
    ),
)


with mlflow.start_run(run_name="chonkie + SemanticChunker + sber_embeds"):
    mlflow.log_param("model", model)
    mlflow.log_param("model_params", embeddings.__dict__)
    mlflow.log_param("collection_name", collection_name)
    mlflow.log_param("chunker_name", str(chunker.__class__))
    mlflow.log_param("chunker_params", chunker.__dict__)

    chunk_lengths = []
    points = []

    for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
        text = row['text']

        for n_chunk, chunk in enumerate(chunker.chunk(text), start=1):
            points.append(models.PointStruct(
                id=len(points) + 1
                , vector=embeddings.encode(chunk.text)
                , payload={"page_id": row["page_id"], "chunk_id": n_chunk, "chunk": chunk.text}
                ))

            chunk_lengths.append(len(chunk.text))


    mlflow.log_metric("total_chunks", len(chunk_lengths))
    mlflow.log_metric("min_chunk_length", min(chunk_lengths))
    mlflow.log_metric("max_chunk_length", max(chunk_lengths))
    mlflow.log_metric("avg_chunk_length", sum(chunk_lengths)/len(chunk_lengths))

    client.upload_points(
        collection_name=collection_name,
        points=points
    )

    def embeddings_function(question: str) -> list[float]:
        return embeddings.encode(question).tolist()

    # --- Оценка разбиения ---
    eval_df, mrr, mrr_ci = evaluate(embeddings_function, q, client, collection_name, len(docs))
    # Сохраняем таблицу как артефакт
    # mlflow.log_table(eval_df, "eval_df.json")
    # Логируем метрику MRR и доверительный интервал
    mlflow.log_metric("MRR", mrr)
    mlflow.log_metric("MRR_lower", float(mrr_ci[0]))
    mlflow.log_metric("MRR_upper", float(mrr_ci[1]))

Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 172/172 [09:25<00:00,  3.29s/it]


🏃 View run chonkie + SemanticChunker + sber_embeds at: http://109.207.175.83:5000/#/experiments/2/runs/128a237efa2345a697e2dc9d61d6881c
🧪 View experiment at: http://109.207.175.83:5000/#/experiments/2


In [ ]:
from chonkie import SemanticChunker, AutoEmbeddings

# model = "ai-forever/ru-en-RoSBERTa"
model = "deepvk/USER2-base"
# model = "sergeyzh/rubert-mini-frida"
embeddings = SentenceTransformer(model)

chunker = SemanticChunker(
    embedding_model = model,
    chunk_size = 256
)

collection_name=str(uuid.uuid1())

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embeddings.get_sentence_embedding_dimension(),  # Vector size is defined by used model
        distance=models.Distance.COSINE,
    ),
)


with mlflow.start_run(run_name="chonkie + SemanticChunker + vk_embeds"):
    mlflow.log_param("model", model)
    mlflow.log_param("model_params", embeddings.__dict__)
    mlflow.log_param("collection_name", collection_name)
    mlflow.log_param("chunker_name", str(chunker.__class__))
    mlflow.log_param("chunker_params", chunker.__dict__)

    chunk_lengths = []
    points = []

    for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
        text = row['text']

        for n_chunk, chunk in enumerate(chunker.chunk(text), start=1):
            points.append(models.PointStruct(
                id=len(points) + 1
                , vector=embeddings.encode(chunk.text)
                , payload={"page_id": row["page_id"], "chunk_id": n_chunk, "chunk": chunk.text}
                ))

            chunk_lengths.append(len(chunk.text))


    mlflow.log_metric("total_chunks", len(chunk_lengths))
    mlflow.log_metric("min_chunk_length", min(chunk_lengths))
    mlflow.log_metric("max_chunk_length", max(chunk_lengths))
    mlflow.log_metric("avg_chunk_length", sum(chunk_lengths)/len(chunk_lengths))

    client.upload_points(
        collection_name=collection_name,
        points=points
    )

    def embeddings_function(question: str) -> list[float]:
        return embeddings.encode(question).tolist()

    # --- Оценка разбиения ---
    eval_df, mrr, mrr_ci = evaluate(embeddings_function, q, client, collection_name, len(docs))
    # Сохраняем таблицу как артефакт
    # mlflow.log_table(eval_df, "eval_df.json")
    # Логируем метрику MRR и доверительный интервал
    mlflow.log_metric("MRR", mrr)
    mlflow.log_metric("MRR_lower", float(mrr_ci[0]))
    mlflow.log_metric("MRR_upper", float(mrr_ci[1]))

100%|██████████| 172/172 [05:48<00:00,  2.03s/it]


🏃 View run chonkie + SemanticChunker + sber_embeds at: http://109.207.175.83:5000/#/experiments/2/runs/fe381e204c21425b8e3fb4494f97e256
🧪 View experiment at: http://109.207.175.83:5000/#/experiments/2


Для памяти оставим

In [ ]:
from chonkie import SemanticChunker, AutoEmbeddings

ids = []
documents = []
embeds = []
metadatas = []

# model = "ai-forever/ru-en-RoSBERTa"
model = "deepvk/USER2-base"
# model = "sergeyzh/rubert-mini-frida"
embeddings = SentenceTransformer(model)

chunker = SemanticChunker(
    embedding_model = model,
    chunk_size = 256
).from_recipe("markdown", lang="en")

for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
    text = row['text']

    for n_chunk, chunk in enumerate(chunker.chunk(text), start=1):
        ids.append(f"id_{n}_{n_chunk}")
        documents.append(chunk.text)
        embeds.append(embeddings.encode(chunk.text))
        metadatas.append({"page_id": row["page_id"], "chunk_id": n_chunk})

with chroma_collection("test_markdown") as test:
    test.add(ids=ids, documents=documents, embeddings=embeds, metadatas=metadatas)
    display(evaluate(embeddings, q, test, len(docs)))

  5%|▍         | 8/172 [00:11<04:11,  1.53s/it]/usr/local/lib/python3.12/dist-packages/chonkie/embeddings/model2vec.py:64: RuntimeWarning: invalid value encountered in divide
  return np.divide(
100%|██████████| 172/172 [01:47<00:00,  1.60it/s]


,question,position,score,position_list
0,"Посещение завершено, как закрыть случай лечения?",2.000,0.5,"[(74, 4), (2, 83), (57, 42), (29, 28), (7, 110..."
1,Как оформить направление на МСЭ?,1.000,1.0,"[(15, 81), (1, 81), (81, 69), (81, 70), (5, 12..."
2,Как создать направление на диагностическое исс...,1.000,1.0,"[(80, 5), (1, 32), (25, 170), (80, 2), (80, 29..."
3,Как создать МКСБ?,1.000,1.0,"[(33, 54), (88, 10), (54, 26), (2, 52), (78, 1..."
4,Как оформить направление на плановую госпитали...,42.000,0.02381,"[(2, 104), (88, 2), (133, 3), (1, 88), (7, 88)..."
5,Как перейти в «План иммунопрофилактики»?,1.000,1.0,"[(6, 8), (8, 118), (8, 4), (8, 33), (8, 1), (7..."
6,Кака создавать реестры в ВебМИС?,2.000,0.5,"[(65, 1), (24, 171), (171, 2), (38, 171), (8, ..."
7,"Какую роль нужно добавить пользователю, чтобы ...",1.000,1.0,"[(30, 35), (20, 30), (29, 30), (30, 5), (25, 3..."
8,Как создать новый МКАБ?,1.000,1.0,"[(1, 31), (2, 168), (31, 14), (31, 81), (70, 1..."
9,Как проверить факт прикрепления пациента в сис...,16.000,0.0625,"[(13, 29), (27, 11), (1, 27), (17, 78), (16, 2..."


for debug


# Chunking markdown-текста с предобработкой с RecursiveChunker
Замена изображеий из markdown-текста и разбиение на чанки с помощью RecursiveChunker.

Для сберовской база 0.64. Для вк база 0.76

In [ ]:
from chonkie import RecursiveChunker

model = "ai-forever/ru-en-RoSBERTa"
# model = "deepvk/USER2-base"
# model = "sergeyzh/rubert-mini-frida"
embeddings = SentenceTransformer(model)

chunker = RecursiveChunker(
    chunk_size = 128
).from_recipe("markdown", lang="en")

collection_name=str(uuid.uuid1())

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embeddings.get_sentence_embedding_dimension(),  # Vector size is defined by used model
        distance=models.Distance.COSINE,
    ),
)


with mlflow.start_run(run_name="chonkie + RecursiveChunker + sber_embeds + preprocess"):
    mlflow.log_param("model", model)
    mlflow.log_param("model_params", embeddings.__dict__)
    mlflow.log_param("collection_name", collection_name)
    mlflow.log_param("chunker_name", str(chunker.__class__))
    mlflow.log_param("chunker_params", chunker.__dict__)

    chunk_lengths = []
    points = []

    for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
        text = row['text']
        clear_text, placeholders = preprocess_images(text)

        for n_chunk, chunk in enumerate(chunker.chunk(text), start=1):
            points.append(models.PointStruct(
                id=len(points) + 1
                , vector=embeddings.encode(chunk.text)
                , payload={"page_id": row["page_id"], "chunk_id": n_chunk, "chunk": chunk.text}
                ))

            chunk_lengths.append(len(chunk.text))


    mlflow.log_metric("total_chunks", len(chunk_lengths))
    mlflow.log_metric("min_chunk_length", min(chunk_lengths))
    mlflow.log_metric("max_chunk_length", max(chunk_lengths))
    mlflow.log_metric("avg_chunk_length", sum(chunk_lengths)/len(chunk_lengths))

    client.upload_points(
        collection_name=collection_name,
        points=points
    )

    def embeddings_function(question: str) -> list[float]:
        return embeddings.encode(question).tolist()

    # --- Оценка разбиения ---
    eval_df, mrr, mrr_ci = evaluate(embeddings_function, q, client, collection_name, len(docs))
    # Сохраняем таблицу как артефакт
    # mlflow.log_table(eval_df, "eval_df.json")
    # Логируем метрику MRR и доверительный интервал
    mlflow.log_metric("MRR", mrr)
    mlflow.log_metric("MRR_lower", float(mrr_ci[0]))
    mlflow.log_metric("MRR_upper", float(mrr_ci[1]))

Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 172/172 [01:07<00:00,  2.56it/s]


🏃 View run chonkie + SemanticChunker + sber_embeds + preprocess at: http://109.207.175.83:5000/#/experiments/2/runs/ae1c7bf8008a48bd8346b639c52f6f3f
🧪 View experiment at: http://109.207.175.83:5000/#/experiments/2


In [ ]:
from chonkie import RecursiveChunker

# model = "ai-forever/ru-en-RoSBERTa"
model = "deepvk/USER2-base"
# model = "sergeyzh/rubert-mini-frida"
embeddings = SentenceTransformer(model)

chunker = RecursiveChunker(
    chunk_size = 128
).from_recipe("markdown", lang="en")

collection_name=str(uuid.uuid1())

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embeddings.get_sentence_embedding_dimension(),  # Vector size is defined by used model
        distance=models.Distance.COSINE,
    ),
)


with mlflow.start_run(run_name="chonkie + RecursiveChunker + vk_embeds + preprocess"):
    mlflow.log_param("model", model)
    mlflow.log_param("model_params", embeddings.__dict__)
    mlflow.log_param("collection_name", collection_name)
    mlflow.log_param("chunker_name", str(chunker.__class__))
    mlflow.log_param("chunker_params", chunker.__dict__)

    chunk_lengths = []
    points = []

    for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
        text = row['text']
        clear_text, placeholders = preprocess_images(text)

        for n_chunk, chunk in enumerate(chunker.chunk(text), start=1):
            points.append(models.PointStruct(
                id=len(points) + 1
                , vector=embeddings.encode(chunk.text)
                , payload={"page_id": row["page_id"], "chunk_id": n_chunk, "chunk": chunk.text}
                ))

            chunk_lengths.append(len(chunk.text))


    mlflow.log_metric("total_chunks", len(chunk_lengths))
    mlflow.log_metric("min_chunk_length", min(chunk_lengths))
    mlflow.log_metric("max_chunk_length", max(chunk_lengths))
    mlflow.log_metric("avg_chunk_length", sum(chunk_lengths)/len(chunk_lengths))

    client.upload_points(
        collection_name=collection_name,
        points=points
    )

    def embeddings_function(question: str) -> list[float]:
        return embeddings.encode(question).tolist()

    # --- Оценка разбиения ---
    eval_df, mrr, mrr_ci = evaluate(embeddings_function, q, client, collection_name, len(docs))
    # Сохраняем таблицу как артефакт
    # mlflow.log_table(eval_df, "eval_df.json")
    # Логируем метрику MRR и доверительный интервал
    mlflow.log_metric("MRR", mrr)
    mlflow.log_metric("MRR_lower", float(mrr_ci[0]))
    mlflow.log_metric("MRR_upper", float(mrr_ci[1]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
100%|██████████| 172/172 [00:41<00:00,  4.14it/s]


🏃 View run chonkie + RecursiveChunker + vk_embeds + preprocess at: http://109.207.175.83:5000/#/experiments/2/runs/5c094bd8722d41f4b0e9bbbbdb2442c8
🧪 View experiment at: http://109.207.175.83:5000/#/experiments/2


# Chunking markdown-текста с предобработкой с SemanticChunker
Замена изображеий из markdown-текста и разбиение на чанки с помощью SemanticChunker.

In [ ]:
from chonkie import SemanticChunker, AutoEmbeddings

model = "ai-forever/ru-en-RoSBERTa"
# model = "deepvk/USER2-base"
# model = "sergeyzh/rubert-mini-frida"
embeddings = SentenceTransformer(model)

chunker = SemanticChunker(
    embedding_model = AutoEmbeddings.get_embeddings(model),
    chunk_size = 256
).from_recipe("markdown", lang="en")


collection_name=str(uuid.uuid1())

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embeddings.get_sentence_embedding_dimension(),  # Vector size is defined by used model
        distance=models.Distance.COSINE,
    ),
)


with mlflow.start_run(run_name="chonkie + SemanticChunker + sber_embeds + preprocess"):
    mlflow.log_param("model", model)
    mlflow.log_param("model_params", embeddings.__dict__)
    mlflow.log_param("collection_name", collection_name)
    mlflow.log_param("chunker_name", str(chunker.__class__))
    mlflow.log_param("chunker_params", chunker.__dict__)

    chunk_lengths = []
    points = []

    for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
        text = row['text']
        clear_text, placeholders = preprocess_images(text)

        for n_chunk, chunk in enumerate(chunker.chunk(text), start=1):
            points.append(models.PointStruct(
                id=len(points) + 1
                , vector=embeddings.encode(chunk.text)
                , payload={"page_id": row["page_id"], "chunk_id": n_chunk, "chunk": chunk.text}
                ))

            chunk_lengths.append(len(chunk.text))


    mlflow.log_metric("total_chunks", len(chunk_lengths))
    mlflow.log_metric("min_chunk_length", min(chunk_lengths))
    mlflow.log_metric("max_chunk_length", max(chunk_lengths))
    mlflow.log_metric("avg_chunk_length", sum(chunk_lengths)/len(chunk_lengths))

    client.upload_points(
        collection_name=collection_name,
        points=points
    )

    def embeddings_function(question: str) -> list[float]:
        return embeddings.encode(question).tolist()

    # --- Оценка разбиения ---
    eval_df, mrr, mrr_ci = evaluate(embeddings_function, q, client, collection_name, len(docs))
    # Сохраняем таблицу как артефакт
    # mlflow.log_table(eval_df, "eval_df.json")
    # Логируем метрику MRR и доверительный интервал
    mlflow.log_metric("MRR", mrr)
    mlflow.log_metric("MRR_lower", float(mrr_ci[0]))
    mlflow.log_metric("MRR_upper", float(mrr_ci[1]))

Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/129M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  5%|▌         | 9/172 [00:12<03:34,  1.31s/it]/usr/local/lib/python3.12/dist-packages/chonkie/embeddings/model2vec.py:64: RuntimeWarning: invalid value encountered in divide
  return np.divide(
100%|██████████| 172/172 [02:01<00:00,  1.42it/s]


🏃 View run chonkie + SemanticChunker + sber_embeds + preprocess at: http://109.207.175.83:5000/#/experiments/2/runs/3e74ff72590a4755a8e74e8734cf2f89
🧪 View experiment at: http://109.207.175.83:5000/#/experiments/2


In [ ]:
from chonkie import SemanticChunker, AutoEmbeddings

# model = "ai-forever/ru-en-RoSBERTa"
model = "deepvk/USER2-base"
# model = "sergeyzh/rubert-mini-frida"
embeddings = SentenceTransformer(model)

chunker = SemanticChunker(
    embedding_model = AutoEmbeddings.get_embeddings(model),
    chunk_size = 256
).from_recipe("markdown", lang="en")


collection_name=str(uuid.uuid1())

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embeddings.get_sentence_embedding_dimension(),  # Vector size is defined by used model
        distance=models.Distance.COSINE,
    ),
)


with mlflow.start_run(run_name="chonkie + SemanticChunker + vk_embeds + preprocess"):
    mlflow.log_param("model", model)
    mlflow.log_param("model_params", embeddings.__dict__)
    mlflow.log_param("collection_name", collection_name)
    mlflow.log_param("chunker_name", str(chunker.__class__))
    mlflow.log_param("chunker_params", chunker.__dict__)

    chunk_lengths = []
    points = []

    for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
        text = row['text']
        clear_text, placeholders = preprocess_images(text)

        for n_chunk, chunk in enumerate(chunker.chunk(text), start=1):
            points.append(models.PointStruct(
                id=len(points) + 1
                , vector=embeddings.encode(chunk.text)
                , payload={"page_id": row["page_id"], "chunk_id": n_chunk, "chunk": chunk.text}
                ))

            chunk_lengths.append(len(chunk.text))


    mlflow.log_metric("total_chunks", len(chunk_lengths))
    mlflow.log_metric("min_chunk_length", min(chunk_lengths))
    mlflow.log_metric("max_chunk_length", max(chunk_lengths))
    mlflow.log_metric("avg_chunk_length", sum(chunk_lengths)/len(chunk_lengths))

    client.upload_points(
        collection_name=collection_name,
        points=points
    )

    def embeddings_function(question: str) -> list[float]:
        return embeddings.encode(question).tolist()

    # --- Оценка разбиения ---
    eval_df, mrr, mrr_ci = evaluate(embeddings_function, q, client, collection_name, len(docs))
    # Сохраняем таблицу как артефакт
    # mlflow.log_table(eval_df, "eval_df.json")
    # Логируем метрику MRR и доверительный интервал
    mlflow.log_metric("MRR", mrr)
    mlflow.log_metric("MRR_lower", float(mrr_ci[0]))
    mlflow.log_metric("MRR_upper", float(mrr_ci[1]))

100%|██████████| 172/172 [01:52<00:00,  1.53it/s]


🏃 View run chonkie + SemanticChunker + vk_embeds + preprocess at: http://109.207.175.83:5000/#/experiments/2/runs/64b465e2e46442cc9dd04f37cc6b0d85
🧪 View experiment at: http://109.207.175.83:5000/#/experiments/2


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Пример: dense (эмбеддинги) + sparse (BM25/TF-IDF)
client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": models.VectorParams(
            size=embeddings.get_sentence_embedding_dimension(),
            distance=models.Distance.COSINE,
        ),
        "sparse": models.VectorParams(
            size=100_000,  # размерность словаря sparse-вектора (пример)
            distance=models.Distance.DOT,
            on_disk=True
        ),
    }
)

# Собираем все чанки для обучения словаря
all_chunks = []
for index, row in docs.iterrows():
    for chunk in RecursiveChunker(chunk_size=128).from_recipe("markdown", lang="en").chunk(row['text']):
        all_chunks.append(chunk.text)
tfidf = TfidfVectorizer(max_features=100_000)
tfidf.fit(all_chunks)

points = []
for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
    text = row['text']
    for n_chunk, chunk in enumerate(chunker.chunk(text), start=1):
        dense_vec = embeddings.encode(chunk.text)
        sparse_vec = tfidf.transform([chunk.text]).toarray()[0]
        # Qdrant ожидает sparse-вектор как словарь: {"indices": [...], "values": [...]}
        sparse_qdrant = {
            "indices": sparse_vec.nonzero()[0].tolist(),
            "values": sparse_vec[sparse_vec != 0].tolist()
        }
        points.append(
            models.PointStruct(
                id=len(points) + 1,
                vector={
                    "dense": dense_vec,
                    "sparse": sparse_qdrant
                },
                payload={
                    "page_id": row["page_id"],
                    "chunk_id": n_chunk,
                    "chunk": chunk.text
                }
            )
        )

client.upload_points(
    collection_name=collection_name,
    points=points
)

# Визуализация чанков SemanticChunker
Визуализация результата разбиения текста на чанки с помощью SemanticChunker и Visualizer.

In [ ]:
from chonkie import SemanticChunker

chunker = SemanticChunker(
    embedding_model = model,
    chunk_size = 256
).from_recipe("markdown", lang="en")

Visualizer()(chunker.chunk(remove_image_tags(docs["text"][6])))

Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Реестры

h1. Реестры ОМС


h2. Описание



В +реестр медицинских услуг в системе ОМС+ включаются все услуги, которые были оказаны пациентам при получении ими 
бесплатной медицинской помощи по программе обязательного медицинского страхования в лечебно-профилактическом 
учреждении. Формирование реестра ОМС осуществляется согласно Генерального тарифного соглашения Территориального 
фонда обязательного медицинского страхования.

h2. Типы реестров



| h1. Наименование реестра* | *Описание* |
| Поликлиника, Стационар | Законченные случаи оказанной медицинской помощи, кроме высокотехнологичной медицинской 
помощи, медицинской помощи по диспансеризации, профилактическим медицинским осмотрам несовершеннолетних и 
профилактическим медицинским осмотрам взрослого населения, медицинской помощи при подозрении на злокачественное 
новообразование или установленном диагнозе злокачественного новообразования |
| ДД ОГВН 1 этап | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках первого
этапа диспансеризации определенных групп взрослого населения; |
| ДД ОГВН 2 этап | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках второго
этапа диспансеризации определенных групп взрослого населения; |
| УД ОГВН 1 этап | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках первого
этапа *углубленной* диспансеризации определенных групп взрослого населения, перенесших COVID-19; |
| УД ОГВН 2 этап | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках второго
этапа *углубленной* диспансеризации определенных групп взрослого населения, перенесших COVID-19; |
| Профосмотры взрослого населения | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу
в рамках профилактических осмотров взрослого населения |
| ДДС | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках диспансеризации 
пребывающих в стационарных учреждениях детей-сирот и детей, находящихся в трудной жизненной ситуации |
| ДДС (опека) | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках 
диспансеризации детей-сирот и детей, оставшихся без попечения родителей, в том числе усыновленных (удочеренных), 
принятых под опеку (попечительство), в приемную или патронатную семью |
| Профилактический медосмотр несовершеннолетних | Законченные случаи на оплату медицинской помощи, оказанной 
застрахованному лицу в рамках профилактических медицинских осмотров несовершеннолетних |
| Поликлиника (ЗНО), Стационар (ЗНО) | Законченные случаи оказанной медицинской помощи при подозрении на 
злокачественное новообразование или установленном диагнозе злокачественного новообразования |
| ВМП | Законченные случаи оказанной высокотехнологичной медицинской помощи |

h2. Виды реестров



*Ежемесячно* подаются реестры на оплату:
* Поликлиника 
* Стационар
* Поликлиника (ЗНО)
* Стационар (ЗНО)
* ДД ОГВН 1 этап
* ДД ОГВН 2 этап
* Профосмотры взрослого населения
* ДДС
* ДДС (опека)
* Профилактический медосмотр несовершеннолетних
* ВМП

*Еженедельно* подаются реестры на оплату:
* УД ОГВН 1 этап
* УД ОГВН 2 этап

*В соответствии с приказом ФФОМС №79 (Приложение Д)* файлы реестров именуются особым образом.
*Пример* наименований файлов реестров, которые *ежемесячно* отправляются от МО в ТФОМС:

Архив HM280012T28_21081.zip, в котором находятся все файлы реестра для ежемесячной подачи:
HM280012T28_21081.xml - Основной (Общий)
DPM280012T28_21081.xml - ДД ОГВН 1 этап
DVM280012T28_21081.xml - ДД ОГВН 2 этап
DOM280012T28_21081.xml - Профосмотры взрослого населения
DSM280012T28_21081.xml - ДДС
DUM280012T28_21081.xml - ДДС (опека)
DFM280012T28_21081.xml - Профилактический медосмотр несовершеннолетних
CM280012T28_21081.xml - ОНКО
TM280012T28_21081.xml - ВМП

Пример наименований файлов реестров, которые еженедельно отправляются от МО в ТФОМС:

Архив DAM280012T28_21081.zip, в котором находятся все файлы реес

# Визуализация чанков RecursiveChunker
Визуализация результата разбиения текста на чанки с помощью RecursiveChunker и Visualizer.

In [ ]:
from chonkie import Visualizer

chunker = RecursiveChunker(
    chunk_size = 128
).from_recipe("markdown", lang="en")

Visualizer()(chunker.chunk(docs.loc[27, 'text']))

WEB Регистратура: Работа с МКАБ в синей версии

h1. Работа с МКАБ в синей версии

МКАБ создается в зеленой МИС ("инструкция по созданию 
МКАБ":https://sd.hostco.ru/projects/amurmis/wiki/%D0%A1%D0%BE%D0%B7%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_%D0%B8_%D1%80%D0%
B5%D0%B4%D0%B0%D0%BA%D1%82%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D0%9C%D0%9A%D0%90%D0%91), чтобы перейти
в редактировании в синей МКАБ, найдите карту нужного пациента, нажмите правой кнопкой мыши и выберете «Посмотреть 
МКАБ».
{{4743cb09-1b1e-49cc-bf49-ddd8a676fa24.bmp}}

В новой вкладке откроется МКАБ выбранного пациента.

{{d3e404e4-3db2-4d0a-a7f1-bbf526adca17.bmp}}

Слева находится панель разделов МКАБ. С её помощью удобно быстро переходить по разделам. 
{{bcdcdba3-104c-4091-92ab-d71b8e635db5.bmp}}


h3. Персональные данные 

Для редактирования персональных данных нажмите на «карандаш» {{691588ee-c33f-47ca-a2a9-c0ab6be62fae.bmp}},  который
расположен рядом с именем пациента.

При редактировании персональных данных есть 4 основных подраздела: 

h1.  Основная информация, где указываются данные пациента.
* Полисы, где отображаются полисы 
* Прикрепления, где отображаются прикрепления пациента по МО
* Дополнительная информация, где указывается доп. сведения по пациенту (сведения о работе и учебе, показатели 
здоровья, представители, согласия и тд)
{{61c19f8d-c8bf-40cf-b123-5a95a925bed3.bmp}}

Поля, помеченные звездочкой {{b64cd6ad-25b6-4c9b-a7df-27ddb25c3b9d.bmp}}  обязательны для заполнения.
В основной информации заполняются данные пациента: Номер МКАБ, ФИО, дата рождения, соц. статус и тд.
{{cb6ff3d5-0989-4528-8f3c-c5afa5a9574a.bmp}}
{{68193685-769d-4a37-889f-a530323a466c.bmp}}
{{0d5543af-1671-4c17-a857-3575e6ed3bf4.bmp}}

После заполнения или редактирования данных нажмите {{1b4c771b-5856-429e-83a4-f91a641dac27.bmp}},  если изменение 
данных не требуется, то {{e77b7313-9e65-4872-ad9c-65b1f8e143f9.bmp}} .


h3. Представитель пациента

Если у пациента есть представители, то добавьте их, нажав {{46c4c73b-95ce-444f-a23a-cfdfa8728f4b.bmp}}, в разделе 
«представители».

Заполнить данные. Если у представителя имеется МКАБ, добавьте её с помощью 
{{dd706119-dbf6-436d-991e-c18f13b80702.bmp}}, найдите представителя и выберете его. После чего данные представителя
заполнятся автоматически. 
Проставить галочки, напротив того, кем является представитель.
{{7d9e7cb5-ef41-43ce-81d3-416208f1ced4.bmp}}

После заполнения информации нажмите {{1b4c771b-5856-429e-83a4-f91a641dac27.bmp}}, для добавления данных, либо 
{{e77b7313-9e65-4872-ad9c-65b1f8e143f9.bmp}}, если сохранение данных не требуется.

h3. Расположение карты

Расположение карты, показывает движение МКАБ. Чтобы указать новые данные по движению карты, нажмите 
{{46c4c73b-95ce-444f-a23a-cfdfa8728f4b.bmp}}, заполните данные и сохраните изменения.

{{995b3154-922c-47e8-ae55-47e33e4e7b9f.bmp}}
{{f5054fca-ca80-45cd-866d-f0d70bad8273.bmp}}
Отправитель указывается автоматически. Есть возможность выбора отправителя вручную.

h3. Прикрепления

Прикрепления, показывает участки прикрепления пациента. Добавьте их, нажав на 
{{46c4c73b-95ce-444f-a23a-cfdfa8728f4b.bmp}}, внесите данные и сохраните. 

{{59f0d02e-6082-4e4e-83de-a399ec11c84c.bmp}}
{{bce1ea44-0643-45c4-b3b6-71f9e0ffc1a6.bmp}}

h3. Дополнительная информация

В дополнительной информации добавьте информацию о потенциально-опасных социальных и/или рабочих факторах, если 
данные были предоставлены.

{{cb8445d7-2746-41d3-a40c-da58abda919d.bmp}}
{{95decb83-ea03-4905-8dbd-96dc82eef3b9.bmp}}

h3. Информация о занятии спортом

Следующий раздел — это информация о занятии спортом.

{{089d6ceb-0dce-43d4-85e5-7633580c38e9.bmp}}

Через {{46c4c73b-95ce-444f-a23a-cfdfa8728f4b.bmp}} внесите данные о занятиях спортом. 
{{beb674de-6f3a-4e52-a996-9dcfc924511b.bmp}}
Заполните информацию, проставьте галки, если это основной спорт и участник соревнований и нажмите 
{{1b4c771b-5856-429e-83a4-f91a641dac27.bmp}}, если нужно отменить всё, то нажмите 
{{e77b7313-9e65-4872-ad9c-65b1f8e143f9.bmp}}. 
После сох

In [ ]:
from chonkie import Visualizer

chunker = RecursiveChunker(
    chunk_size = 128
).from_recipe("markdown", lang="en")

Visualizer()(chunker.chunk(preprocess_images(preprocess_headings(docs.loc[0, 'text']))[0]))
# preprocess_images(docs.loc[27, 'text'])[1]

WEB Диспансеризация


# Диспансеризация в web-версии МИС



## Настройка ролей

Перед непосредственной работой по оформлению медосмотров и диспансеризаций пациентов должна быть осуществлена 
настройка системы в части медицинских обследований.
Для настройки данного модуля, необходимо назначить ответственному лицу роли: 
 -  Региональный администратор (Диспансеризация)
 -  Медицинские обследования (Администрирование)
 -  Медицинские обследования
Для создания маршрутных листов регистратором или кабинетом Профилактики и работе врача с картой мед.обследования 
назначить роль «Медицинские обследования».


## Настройка модуля «Профилактика»

Настройка модуля «Профилактика» начинается с раздела «Диспансеризация»[IMG_1]

Сопоставьте мероприятия с ресурсами (кабинеты, врачи, оборудование), которые будут выполнять мероприятия. Для этого
переходим в раздел «Мероприятия и ресурсы»
Загрузится страница сопоставления, где вы осуществляете поиск по «Виду медицинского обследования» и нажимаете 
кнопку «Найти[IMG_2]

Например, произведем настройку для «Водительской справки А, В, М»
После нажатия на кнопку «Найти» появится список всех мероприятий[IMG_3]

Теперь производим сопоставление каждого мероприятия с ресурсом через кнопку редактировать:[IMG_4]

Вводим начальные символы ресурса и выбираем ресурс, после всего выбора обязательно нажмите кнопку 
«Сохранить»[IMG_5]

Если ресурса нет, то его не указываете, далее можно указать, что выполнено ранее в другом МО.

# Обязательное указание ресурса для мероприятий с видом "Анкетирование и Прием врача"* (Смена вида мероприятия 
осуществляется по заявке сотрудниками Хост).


## Создание расписания для записи на медосмотры и диспансеризацию

После сопоставления всех мероприятий с ресурсами, необходимо создать расписание на каждый ресурс с типами 
«Медосмотр» (для записи на медицинский осмотр) и «Диспансеризация» (для записи на диспансеризацию)[IMG_6]


## Формирование маршрутного листа и запись в расписание

Регистратор или ответственное лицо создает «Маршрутный лист» в расписании на прием[IMG_7]

Находят пациента по ФИО/СНИЛС/Полису/Номеру карты и нажимают кнопку «Выбрать»[IMG_8]

Открывается окно для формирования «Маршрутного листа», где выбирается «План» и период прохождения 
мероприятий[IMG_9]

*Важно! По периоду система ищет свободные слоты в расписании ресурсов.*

Далее нажимаете кнопку «Подобрать мероприятия»[IMG_10] После чего будет сформирован список мероприятий по 
выбранному плану.

Если мероприятие было сделано ранее, то нажмите кнопку[IMG_11] и проставьте дату прохождения.[IMG_12]

Далее вы формируете маршрутный лист по кнопке[IMG_13] В маршрутном листе все мероприятия распределяются по ресурсам
на доступное время. При желании можно запись перенести.[IMG_14]

В итоге вы печатаете маршрутный лист по кнопке «Печать»[IMG_15]

Далее нажимаете «Сохранить и закрыть».


## Формирование маршрутного листа без записи в расписание

Создать Маршрутный лист можно без привязки к расписанию. Для этого на главном экране заходим в раздел "Расписание 
приема"[IMG_16]

В боковом меню выбираем «Маршрутный лист»[IMG_17]

В открывшемся окне находим нужного нам пациента и нажимаем кнопку «Выбрать»[IMG_18]

Откроется окно создания маршрутного листа, в котором необходимо выбрать План (1), модель (2) и нажать кнопку 
Подобрать мероприятие (3)[IMG_19]

Далее откроется маршрутный лист со всеми мероприятиями, после чего нажмите кнопку «сформировать маршрутный 
лист»[IMG_20]

Сформируется маршрутный лист, напротив мероприятия будет указан тип записи «Самозапись»[IMG_21]

Нажмите кнопку «Сохранить и закрыть». 
Далее для работы с картой перейдите в блок «Карты медицинских обследований»[IMG_22]

В открывшемся списке выберите нужную карту[IMG_23]

Далее процесс заполнения карты не отличается от заполнения карты с привязкой к расписанию. 


## Перенести мероприятие на другое время или изменить ресурс

1. Открываем Маршрутный лист
2. На необходимом мероприятии нажимаем на кнопку "Перенести запись" 
3. В открывшейся форме перено

In [ ]:
from chonkie import RecursiveChunker

ids = []
documents = []
embeds = []
metadatas = []

chunker = RecursiveChunker(
    chunk_size = 128
).from_recipe("markdown", lang="en")

for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
    text = row['text']
    clear_text, placeholders = preprocess_images(text)

    for n_chunk, chunk in enumerate(chunker.chunk(clear_text), start=1):
        ids.append(f"id_{n}_{n_chunk}")
        documents.append(chunk.text)
        embeds.append(embeddings.encode(chunk.text))
        metadatas.append({"page_id": row["page_id"], "chunk_id": n_chunk})



100%|██████████| 172/172 [17:49<00:00,  6.22s/it]


In [ ]:
with chroma_collection("result_view") as view:
    view.add(ids=ids, documents=documents, embeddings=embeds, metadatas=metadatas)
    result = view.query(embeddings.encode(q.loc[9, "question"]), n_results=50)
    print(result)

{'ids': [['id_27_1', 'id_31_2', 'id_29_1', 'id_27_2', 'id_47_1', 'id_88_2', 'id_27_4', 'id_56_1', 'id_4_2', 'id_56_3', 'id_41_12', 'id_78_2', 'id_56_7', 'id_23_1', 'id_29_2', 'id_38_1', 'id_21_3', 'id_2_1', 'id_15_1', 'id_55_1', 'id_4_3', 'id_54_4', 'id_19_2', 'id_1_3', 'id_166_3', 'id_54_1', 'id_118_7', 'id_25_2', 'id_14_1', 'id_2_11', 'id_134_2', 'id_41_2', 'id_69_2', 'id_2_10', 'id_26_1', 'id_11_2', 'id_40_2', 'id_3_1', 'id_28_8', 'id_78_1', 'id_2_7', 'id_6_1', 'id_50_1', 'id_89_2', 'id_9_1', 'id_87_6', 'id_46_2', 'id_54_3', 'id_163_2', 'id_42_1']], 'embeddings': None, 'documents': [['WEB Регистратура: Прикрепление через ЕПГУ и по личному заявлению пациента в МО\n\nh1. Прикрепление через ЕПГУ и по личному заявлению пациента в МО\n\n\nДля работы с заявлениями на прикрепление, полученных с Единого портала государственных услуг (ЕПГУ), назначьте пользователю роли: \nh1.  Оператор рассмотрения заявлений на прикрепление \n* Отклонение заявок на прикрепление\n\nРоль добавляется через блок

In [ ]:
display(result['metadatas'][0][38], result['metadatas'][0][1])
display(result['distances'][0][38], result['distances'][0][1])

{'chunk_id': 8, 'page_id': 28}

{'page_id': 31, 'chunk_id': 2}

0.7670572996139526

0.611464262008667

# Внезапно захотел подобрать гиперпараметры

In [ ]:
# Multi-objective: минимизировать близость к док.28 и максимизировать близость к док.26
import optuna
import requests
from sklearn.metrics.pairwise import cosine_distances
from chonkie import RecursiveChunker, SemanticChunker, AutoEmbeddings

config_url = "https://huggingface.co/deepvk/USER2-base/resolve/main/config_sentence_transformers.json"
config = requests.get(config_url).json()
prompt_names = list(config.get("prompts", {}).keys())
prompt_names.append(None)

question = q.loc[9, "question"]  # 10-й вопрос (визуальный, но используем .loc[9])
doc_idx_pos = 27  # 28-й документ: минимизируем расстояние
doc_idx_far = 26  # 27-й документ: максимизируем расстояние
document_pos = docs.loc[doc_idx_pos, "text"]
document_far = docs.loc[doc_idx_far, "text"]

# chunker = RecursiveChunker(
#     chunk_size = 128
# ).from_recipe("markdown", lang="en")
# chunks_pos = [chunk.text for chunk in chunker.chunk(preprocess_images(document_pos)[0])]
# chunks_far = [chunk.text for chunk in chunker.chunk(preprocess_images(document_far)[0])]

def objective(trial):
    prompt_name_q = trial.suggest_categorical("prompt_name_q", prompt_names)
    prompt_name_d = trial.suggest_categorical("prompt_name_d", prompt_names)
    prompt_name_c = trial.suggest_categorical("prompt_name_c", prompt_names)

    chunker = SemanticChunker(
        embedding_model = AutoEmbeddings.get_embeddings(model, default_prompt_name=prompt_name_c),
        chunk_size = 256
    ).from_recipe("markdown", lang="en")
    chunks_pos = [chunk.text for chunk in chunker.chunk(preprocess_images(document_pos)[0])]
    chunks_far = [chunk.text for chunk in chunker.chunk(preprocess_images(document_far)[0])]
    q_emb = embeddings.encode(question, prompt_name=prompt_name_q)

    d_embs_pos = [embeddings.encode(ch, prompt_name=prompt_name_d) for ch in chunks_pos]
    dists_pos = [cosine_distances([q_emb], [d_emb])[0][0] for d_emb in d_embs_pos]
    min_dist_pos = min(dists_pos)  # хотим МИНИМИЗИРОВАТЬ

    d_embs_far = [embeddings.encode(ch, prompt_name=prompt_name_d) for ch in chunks_far]
    dists_far = [cosine_distances([q_emb], [d_emb])[0][0] for d_emb in d_embs_far]
    min_dist_far = min(dists_far)  # хотим МАКСИМИЗИРОВАТЬ

    return min_dist_pos, min_dist_far

study = optuna.create_study(directions=["minimize", "maximize"])
study.optimize(objective, n_trials=100, show_progress_bar=True)

# Выберем трейл с минимальным первым значением и максимальным вторым
best_trial = sorted(study.best_trials, key=lambda t: (t.values[0], -t.values[1]))[0]
print("Лучшие параметры:", best_trial.params)
print("min_dist(док.28):", best_trial.values[0], "| min_dist(док.26):", best_trial.values[1])

[I 2025-10-29 08:57:02,092] A new study created in memory with name: no-name-c9a8c2aa-e871-47f5-b327-ea72eacf8f88


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-10-29 08:57:08,249] Trial 0 finished with values: [0.1961572766304016, 0.1973099708557129] and parameters: {'prompt_name_q': 'classification', 'prompt_name_d': 'search_query', 'prompt_name_c': None}.


[I 2025-10-29 08:57:14,577] Trial 1 finished with values: [0.1898401379585266, 0.19985991716384888] and parameters: {'prompt_name_q': 'search_document', 'prompt_name_d': 'search_document', 'prompt_name_c': 'classification'}.


[I 2025-10-29 08:57:20,576] Trial 2 finished with values: [0.25855445861816406, 0.2441551685333252] and parameters: {'prompt_name_q': 'classification', 'prompt_name_d': 'clustering', 'prompt_name_c': 'search_document'}.


...

[I 2025-10-29 09:07:20,313] Trial 94 finished with values: [0.3448098301887512, 0.32790350914001465] and parameters: {'prompt_name_q': 'search_query', 'prompt_name_d': None, 'prompt_name_c': 'classification'}.
[I 2025-10-29 09:07:26,358] Trial 95 finished with values: [0.1865289807319641, 0.19502758979797363] and parameters: {'prompt_name_q': 'search_document', 'prompt_name_d': 'search_query', 'prompt_name_c': None}.


[I 2025-10-29 09:07:32,816] Trial 96 finished with values: [0.1865289807319641, 0.19502758979797363] and parameters: {'prompt_name_q': 'search_document', 'prompt_name_d': 'search_query', 'prompt_name_c': 'clustering'}.


[I 2025-10-29 09:07:39,613] Trial 97 finished with values: [0.312575101852417, 0.30228012800216675] and parameters: {'prompt_name_q': 'search_document', 'prompt_name_d': 'clustering', 'prompt_name_c': 'classification'}.


[I 2025-10-29 09:07:46,597] Trial 98 finished with values: [0.1865289807319641, 0.19502758979797363] and parameters: {'prompt_name_q': 'search_document', 'prompt_name_d': 'search_query', 'prompt_name_c': 'clustering'}.


[I 2025-10-29 09:07:52,720] Trial 99 finished with values: [0.3081209659576416, 0.2851814031600952] and parameters: {'prompt_name_q': None, 'prompt_name_d': 'clustering', 'prompt_name_c': 'search_query'}.
Лучшие параметры: {'prompt_name_q': 'search_query', 'prompt_name_d': 'search_query', 'prompt_name_c': 'search_document'}
min_dist(док.28): 0.17090833187103271 | min_dist(док.26): 0.17906618118286133


In [ ]:
import optuna.visualization

optuna.visualization.plot_param_importances(study).show()

# Хочется посмотреть на размерность чанков

In [ ]:
from chonkie import SemanticChunker

chunker = RecursiveChunker(
    chunk_size = 128
).from_recipe("markdown", lang="en")

# Находим самый большой чанк по token_count
largest_chunk = max(
    (chunk for text in docs['text'] for chunk in chunker.chunk(preprocess_images(text)[0])),
    key=lambda chunk: chunk.token_count
)

print(f"Самый большой чанк содержит {largest_chunk.token_count} токенов.")
print("Содержимое самого большого чанка:")
print(largest_chunk.text)

Самый большой чанк содержит 2047 токенов.
Содержимое самого большого чанка:
* «Серия документа» – заполняется вручную с клавиатуры.
* «Номер документа» – заполняется вручную с клавиатуры.
* «Дата выдачи» – заполняется вручную с клавиатуры или путем выбора значения из календаря. Дата выдачи не может быть больше текущей. Поле обязательно для заполнения.
* «Вид документа» – заполняется путем выбора нужного значения из выпадающего списка. Поле обязательно для заполнения.
* «Описание документа» – заполняется вручную с клавиатуры.[IMG_110]

Во вкладке «Другие связанные документы» при добавлении реквизитов бумажных документов доступна возможность прикрепления файла с компьютера.
Для того чтобы прикрепить файл, необходимо нажать кнопку «Прикрепить файл».[IMG_111]

После нажатия кнопки «Прикрепить файл» откроется проводник для выбора прикрепляемого файла. Доступна загрузка только одного файла для одного связанного документа. После выбора файла кнопка «Прикрепить файл» изменится на «Открепить фа

In [ ]:
from chonkie import SemanticChunker

chunker = SemanticChunker(
    embedding_model = model,
    chunk_size = 256
).from_recipe("markdown", lang="en")

# Находим самый большой чанк по token_count
largest_chunk = max(
    (chunk for text in docs['text'] for chunk in chunker.chunk(preprocess_images(text)[0])),
    key=lambda chunk: chunk.token_count
)

print(f"Самый большой чанк содержит {largest_chunk.token_count} токенов.")
print("Содержимое самого большого чанка:")
print(largest_chunk.text)

Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/chonkie/embeddings/model2vec.py:64: RuntimeWarning: invalid value encountered in divide
  return np.divide(


Самый большой чанк содержит 1918 токенов.
Содержимое самого большого чанка:
Наименование реестра* | *Описание* |
| Поликлиника, Стационар | Законченные случаи оказанной медицинской помощи, кроме высокотехнологичной медицинской помощи, медицинской помощи по диспансеризации, профилактическим медицинским осмотрам несовершеннолетних и профилактическим медицинским осмотрам взрослого населения, медицинской помощи при подозрении на злокачественное новообразование или установленном диагнозе злокачественного новообразования |
| ДД ОГВН 1 этап | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках первого этапа диспансеризации определенных групп взрослого населения; |
| ДД ОГВН 2 этап | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках второго этапа диспансеризации определенных групп взрослого населения; |
| УД ОГВН 1 этап | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках первого э